<a href="https://colab.research.google.com/github/Tomca765/Tools-for-the-Analysis-of-Histological-Images-of-Liver-Scaffolds-in-the-QuPath-Environment/blob/main/qupath_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required dependencies
!pip install -q flask opencv-python-headless pillow pyngrok

import threading
import cv2
import numpy as np
from flask import Flask, request, jsonify
from pyngrok import ngrok

# =========================================================================
# CONFIGURATION
# =========================================================================
NGROK_AUTH_TOKEN = "3KdNGUA1jz7IBxOfwhLcWXT7moa_36tNzYBAc1rpuoqap4dAX"  # Paste your token here
NGROK_DOMAIN     = "upstage-justice-oblong.ngrok-free.dev"

app = Flask(__name__)

@app.route('/analyze', methods=['POST'])
def analyze_image():
    # Receive raw image bytes from QuPath
    img_bytes = request.data
    if not img_bytes:
        return jsonify({'status': 'error', 'message': 'No image data received'}), 400

    # Decode bytes to OpenCV BGR image
    nparr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    if img is None:
        return jsonify({'status': 'error', 'message': 'Failed to decode image'}), 400

    # -------------------------------------------------------------
    # OPENCV EDGE & CONTOUR DETECTION
    # -------------------------------------------------------------
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Otsu thresholding to segment structures/edges
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Extract contour boundaries
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    polygons = []
    for cnt in contours:
        # Filter out tiny noise
        if cv2.contourArea(cnt) < 20:
            continue

        # Simplify polygon curve to reduce node count
        epsilon = 0.005 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)

        # Convert contour points into a list of [x, y] coordinates
        pts = [[int(pt[0][0]), int(pt[0][1])] for pt in approx]

        if len(pts) >= 3:
            polygons.append(pts)

    print(f"[Colab Processed] Found {len(polygons)} outlines in incoming region.")

    return jsonify({
        'status': 'success',
        'contour_count': len(polygons),
        'contours': polygons
    })

# Reset existing tunnels on re-run
ngrok.kill()

# Start Flask server
def run_flask():
    app.run(host='0.0.0.0', port=5000)

threading.Thread(target=run_flask, daemon=True).start()






# Connect ngrok tunnel using the 'url' parameter
import os

# Remove old ngrok configuration file if it exists to allow re-running cells safely
ngrok_config = os.path.expanduser("~/.config/ngrok/ngrok.yml")
if os.path.exists(ngrok_config):
    os.remove(ngrok_config)

# Connect ngrok tunnel
ngrok.set_auth_token(NGROK_AUTH_TOKEN)




tunnel = ngrok.connect(5000, url=NGROK_DOMAIN)
colab_url = f"{tunnel.public_url}/analyze"

print("\n" + "=" * 60)
print(f"PERMANENT ENDPOINT ACTIVE: {colab_url}")
print("=" * 60 + "\n")

KeyboardInterrupt: 

In [ ]:
# Install required dependencies

!pip install -q flask opencv-python-headless pillow pyngrok


import os

import threading

import cv2

import numpy as np

from flask import Flask, request, jsonify

from pyngrok import ngrok


# =========================================================================

# CONFIGURATION & TISSUE-AWARE CELL DETECTOR PARAMETERS

# =========================================================================

NGROK_AUTH_TOKEN = "3KdNGUA1jz7IBxOfwhLcWXT7moa_36tNzYBAc1rpuoqap4dAX"

NGROK_DOMAIN     = "upstage-justice-oblong.ngrok-free.dev"


# --- 1. Tissue vs Background (Glass) Masking ---

# Pixels brighter than this value are treated as empty glass/background and ignored.

TISSUE_MAX_BRIGHTNESS = 245   # Range: 200 - 245 (Lower ignores faint tissue; Higher includes more background)

MIN_TISSUE_SATURATION = 20    # Range: 5 - 30 (Filters out uncolored/white glass space)


# --- 2. Cell Segmentation Parameters ---

MIN_CELL_AREA    = 20000        # Minimum size of an internal cell/vacuole (in pixels)

MAX_CELL_AREA    = 15000000000000000000      # Maximum size to prevent capturing full tissue borders

MIN_CIRCULARITY  = 0.0001       # Shape filter (0.0 = long boundary, 1.0 = perfect circle)

BLUR_KERNEL_SIZE = 9          # Smoothing kernel size (must be odd: 3, 5, 7...)

POLYGON_EPSILON  = 0.0006      # Polygon smoothing factor


# Use Adaptive Thresholding to handle local illumination across large tissue regions

USE_ADAPTIVE_THRESH = True

ADAPTIVE_BLOCK_SIZE = 31      # Local pixel neighborhood (must be odd integer)

ADAPTIVE_C          = 3       # Constant subtracted from local mean


app = Flask(__name__)


@app.route('/analyze', methods=['POST'])

def analyze_image():

    img_bytes = request.data

    if not img_bytes:

        return jsonify({'status': 'error', 'message': 'No image data received'}), 400


    nparr = np.frombuffer(img_bytes, np.uint8)

    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)


    if img is None:

        return jsonify({'status': 'error', 'message': 'Failed to decode image'}), 400


    # -------------------------------------------------------------

    # STEP 1: ISOLATE TISSUE FROM GLASS / BACKGROUND

    # -------------------------------------------------------------

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    saturation = hsv[:, :, 1]


    # Create mask where tissue exists (not pure white glass, and has some color/stain)

    tissue_mask = (gray < TISSUE_MAX_BRIGHTNESS) | (saturation > MIN_TISSUE_SATURATION)

    tissue_mask = tissue_mask.astype(np.uint8) * 255


    # Clean tissue mask boundaries

    kernel_tissue = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

    tissue_mask = cv2.morphologyEx(tissue_mask, cv2.MORPH_CLOSE, kernel_tissue)


    # -------------------------------------------------------------

    # STEP 2: CELL DETECTION INSIDE TISSUE ONLY

    # -------------------------------------------------------------

    blurred = cv2.GaussianBlur(gray, (BLUR_KERNEL_SIZE, BLUR_KERNEL_SIZE), 0)


    if USE_ADAPTIVE_THRESH:

        # Adaptive thresholding handles variations across wide slide regions

        binary = cv2.adaptiveThreshold(

            blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,

            cv2.THRESH_BINARY, ADAPTIVE_BLOCK_SIZE, ADAPTIVE_C

        )

    else:

        _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)


    # Restrict binary cell detection STRICTLY inside tissue boundaries

    cells_only_binary = cv2.bitwise_and(binary, binary, mask=tissue_mask)


    # Morphological cleanup inside cell structures

    kernel_cells = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

    cleaned_cells = cv2.morphologyEx(cells_only_binary, cv2.MORPH_OPEN, kernel_cells)


    # Extract contours

    contours, _ = cv2.findContours(cleaned_cells, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)


    polygons = []

    for cnt in contours:

        area = cv2.contourArea(cnt)

        perimeter = cv2.arcLength(cnt, True)



        if perimeter == 0:

            continue



        circularity = (4 * np.pi * area) / (perimeter ** 2)


        # Apply area and shape criteria

        if MIN_CELL_AREA <= area <= MAX_CELL_AREA and circularity >= MIN_CIRCULARITY:

            approx = cv2.approxPolyDP(cnt, POLYGON_EPSILON * perimeter, True)

            pts = [[int(pt[0][0]), int(pt[0][1])] for pt in approx]



            if len(pts) >= 3:

                polygons.append(pts)


    print(f"[Colab Processed] Identified {len(polygons)} internal cells inside tissue.")


    return jsonify({

        'status': 'success',

        'contour_count': len(polygons),

        'contours': polygons

    })


# =========================================================================

# NGROK TUNNEL SETUP

# =========================================================================

ngrok.kill()


ngrok_config = os.path.expanduser("~/.config/ngrok/ngrok.yml")

if os.path.exists(ngrok_config):

    os.remove(ngrok_config)


ngrok.set_auth_token(NGROK_AUTH_TOKEN)


def run_flask():

    app.run(host='0.0.0.0', port=5000)


threading.Thread(target=run_flask, daemon=True).start()


tunnel = ngrok.connect(5000, url=NGROK_DOMAIN)

colab_url = f"{tunnel.public_url}/analyze"


print("\n" + "=" * 60)

print(f"PERMANENT ENDPOINT ACTIVE: {colab_url}")

print("=" * 60 + "\n")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



PERMANENT ENDPOINT ACTIVE: https://upstage-justice-oblong.ngrok-free.dev/analyze



In [1]:
!pip install -q flask opencv-python-headless pillow pyngrok

import os
import threading
import cv2
import numpy as np
from flask import Flask, request, jsonify
from pyngrok import ngrok

# =========================================================================
# CONFIGURATION
# =========================================================================
NGROK_AUTH_TOKEN = "3KdNGUA1jz7IBxOfwhLcWXT7moa_36tNzYBAc1rpuoqap4dAX"
NGROK_DOMAIN     = "upstage-justice-oblong.ngrok-free.dev"

TISSUE_MAX_BRIGHTNESS = 245
MIN_TISSUE_SATURATION = 20

MIN_CELL_AREA     = 20000
MAX_CELL_AREA     = 15000000000000000000
MIN_CIRCULARITY   = 0.0001
BLUR_KERNEL_SIZE  = 9
POLYGON_EPSILON   = 0.0006

USE_ADAPTIVE_THRESH = True
ADAPTIVE_BLOCK_SIZE = 31
ADAPTIVE_C          = 3

app = Flask(__name__)

@app.route('/analyze', methods=['POST'])
def analyze_image():
    img_bytes = request.data
    if not img_bytes:
        return jsonify({'status': 'error', 'message': 'No image data'}), 400

    nparr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if img is None:
        return jsonify({'status': 'error', 'message': 'Decode failed'}), 400

    # 1. TISSUE & PINK COLOR MASKS
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    saturation = hsv[:, :, 1]

    tissue_mask = ((gray < TISSUE_MAX_BRIGHTNESS) | (saturation > MIN_TISSUE_SATURATION)).astype(np.uint8) * 255
    kernel_tissue = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    tissue_mask = cv2.morphologyEx(tissue_mask, cv2.MORPH_CLOSE, kernel_tissue)

    # Detekce růžové (Eosin): Hue 135-175 a 0-10
    pink1 = cv2.inRange(hsv, np.array([135, 30, 50], dtype=np.uint8), np.array([175, 255, 255], dtype=np.uint8))
    pink2 = cv2.inRange(hsv, np.array([0, 30, 50], dtype=np.uint8), np.array([10, 255, 255], dtype=np.uint8))
    pink_mask = cv2.bitwise_or(pink1, pink2)

    # 2. CELL SEGMENTATION
    blurred = cv2.GaussianBlur(gray, (BLUR_KERNEL_SIZE, BLUR_KERNEL_SIZE), 0)
    if USE_ADAPTIVE_THRESH:
        binary = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, ADAPTIVE_BLOCK_SIZE, ADAPTIVE_C)
    else:
        _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    cells_only = cv2.bitwise_and(binary, binary, mask=tissue_mask)
    kernel_cells = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned_cells = cv2.morphologyEx(cells_only, cv2.MORPH_OPEN, kernel_cells)

    contours, _ = cv2.findContours(cleaned_cells, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    polygons = []
    pink_percentages = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue

        circularity = (4 * np.pi * area) / (perimeter ** 2)

        if MIN_CELL_AREA <= area <= MAX_CELL_AREA and circularity >= MIN_CIRCULARITY:
            approx = cv2.approxPolyDP(cnt, POLYGON_EPSILON * perimeter, True)
            pts = [[int(pt[0][0]), int(pt[0][1])] for pt in approx]

            if len(pts) >= 3:
                # Spočítání růžových pixelů uvnitř polygonu buňky
                x, y, w, h = cv2.boundingRect(cnt)
                cell_mask_roi = np.zeros((h, w), dtype=np.uint8)
                cnt_offset = (cnt - [x, y]).astype(np.int32)
                cv2.drawContours(cell_mask_roi, [cnt_offset], -1, 255, thickness=cv2.FILLED)

                total_pixels = cv2.countNonZero(cell_mask_roi)
                if total_pixels > 0:
                    roi_pink = pink_mask[y:y+h, x:x+w]
                    pink_pixels = cv2.countNonZero(cv2.bitwise_and(roi_pink, roi_pink, mask=cell_mask_roi))
                    pink_pct = round((float(pink_pixels) / float(total_pixels)) * 100.0, 2)
                else:
                    pink_pct = 0.0

                polygons.append(pts)
                pink_percentages.append(pink_pct)

    return jsonify({
        'status': 'success',
        'contour_count': len(polygons),
        'contours': polygons,
        'pink_percentages': pink_percentages
    })

# SERVER LAUNCH
ngrok.kill()
ngrok_config = os.path.expanduser("~/.config/ngrok/ngrok.yml")
if os.path.exists(ngrok_config):
    os.remove(ngrok_config)

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000), daemon=True).start()

tunnel = ngrok.connect(5000, url=NGROK_DOMAIN)
print(f"PERMANENT ENDPOINT ACTIVE: {tunnel.public_url}/analyze")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


PERMANENT ENDPOINT ACTIVE: https://upstage-justice-oblong.ngrok-free.dev/analyze
